# 02 — Train baseline + hierarchical models (multiclass outcomes)

This notebook trains:
- A **baseline multiclass boosted model** (LightGBM) on 4 classes.
- A **hierarchical model** (Stage1 NoDementia vs Dementia; Stage2 subtype among dementia cases) and combines probabilities.

It also saves feature importance and subgroup metrics.

In [30]:
# import sys, site, subprocess
# print("Python exe:", sys.executable)
# print("Python version:", sys.version)
# print("User site:", site.getusersitepackages())

# # what pip does THIS python use?
# subprocess.run([sys.executable, "-m", "pip", "--version"])


In [31]:
# import sys, subprocess

# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", "lightgbm"])


In [32]:
#!pip install lightgbm

In [33]:

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Optional (only used if installed)
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM not available; install lightgbm to run boosted models. Error:", repr(e))

RANDOM_STATE = 42

OUT_DIR = "../output"
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

COHORT_PATH = os.path.join(OUT_DIR, "cohort.csv")  # produced by 01_build_cohort
df = pd.read_csv(COHORT_PATH, low_memory=False)

print("Loaded cohort rows:", len(df))
print(df["label"].value_counts(dropna=False))


Loaded cohort rows: 38341
label
Censored         23300
NoDementia_5y    11083
AD                3013
MixedOrOther       592
PD_LBD             289
Vascular            64
Name: count, dtype: int64


In [34]:

# Keep only the 4 non-censored classes for modeling
KEEP = ["NoDementia_5y", "AD", "PD_LBD", "Vascular"]
df4 = df[df["label"].isin(KEEP)].copy()
df4["label"] = df4["label"].astype(str)

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Separate labels and features
drop_cols = [c for c in ["NACCID", "baseline_date", "label", "NACCUDSD"] if c in df4.columns]
y = df4["label"].astype(str)
X_all = df4.drop(columns=drop_cols)


4-class rows: 14449
label
NoDementia_5y    11083
AD                3013
PD_LBD             289
Vascular            64
Name: count, dtype: int64


In [35]:

# --- Treat certain coded vars as categorical (even if stored as ints) ---
CAT_CODE_VARS = [c for c in ["SEX","RACE","HISPANIC"] if c in X_all.columns]

# Standard missing codes
STD_MISS = {-4, 9}

# Clean missing codes and preserve NaNs
for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    s = s.mask(s.isin(list(STD_MISS)), np.nan)
    # RACE sometimes uses 99 for unknown
    if c == "RACE":
        s = s.mask(s.isin([99]), np.nan)
    X_all[c] = s

# Convert to pandas nullable Int + string so OneHot treats as categorical but NaNs stay NaN
for c in CAT_CODE_VARS:
    X_all[c] = X_all[c].astype("Int64").astype("string")

# Education: keep numeric, plus an education bin for subgroup analysis (optional)
if "EDUC" in X_all.columns:
    educ_num = pd.to_numeric(X_all["EDUC"], errors="coerce")
    X_all["EDUC_BIN"] = pd.Series(np.where(educ_num <= 12, "<=12", ">12"), index=X_all.index).astype("string")

# Check what scikit sees as categorical vs numeric
cat_cols = [c for c in X_all.columns if (X_all[c].dtype == "object") or str(X_all[c].dtype).startswith("string")]
num_cols = [c for c in X_all.columns if c not in cat_cols]

print("Cat cols:", cat_cols)
print("Num cols:", len(num_cols))


Cat cols: ['SEX', 'RACE', 'HISPANIC', 'EDUC_BIN']
Num cols: 42


In [36]:

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

print("Train rows:", len(X_train), "Test rows:", len(X_test))


Train rows: 11559 Test rows: 2890


In [37]:

import numpy as np
import pandas as pd

# -----------------------------
# Baseline multiclass boosted model (Option B)
# -----------------------------
custom_w = {
    "NoDementia_5y": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0,
}


def make_sample_weight(y_series, weight_map, default_weight=1.0):
    # map -> float array, and fill any unknown labels safely
    w = y_series.map(weight_map)

    # If any labels missing in the map, fill them (and print them once)
    missing = y_series[w.isna()].unique()
    if len(missing) > 0:
        print("WARNING: labels missing from weight_map; using default_weight for:", list(missing))
        w = w.fillna(default_weight)

    # convert to clean numpy float array (no pd.NA)
    return np.asarray(w, dtype=np.float64)

w_train = make_sample_weight(y_train, custom_w, default_weight=1.0)

# sanity checks (optional but recommended)
assert len(w_train) == len(y_train)
assert np.isfinite(w_train).all()
# w_train = make_sample_weight(y_train, custom_w)

# Label-encode for LightGBM to avoid "Invalid classes" error
label_order = ["AD","NoDementia_5y","PD_LBD","Vascular"]
label_to_int = {lab:i for i,lab in enumerate(label_order)}
int_to_label = {i:lab for lab,i in label_to_int.items()}

# Validate labels
bad_train = sorted(set(y_train.unique()) - set(label_order))
bad_test  = sorted(set(y_test.unique()) - set(label_order))
if bad_train or bad_test:
    raise ValueError(f"Unexpected labels. train={bad_train}, test={bad_test}")

y_train_i = y_train.map(label_to_int).astype(int)
y_test_i  = y_test.map(label_to_int).astype(int)

if HAS_LGBM:
    booster = LGBMClassifier(
        objective="multiclass",
        num_class=4,
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
    )

    clf_boost = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", booster),
    ])


    def sklearn_safe_df(df: pd.DataFrame) -> pd.DataFrame:
        """Convert pandas nullable dtypes (StringDtype/Int64) that contain pd.NA into sklearn-safe object/float with np.nan."""
        out = df.copy()

        for c in out.columns:
            # If it's pandas nullable string dtype -> convert to object and replace <NA> with np.nan
            if str(out[c].dtype).startswith("string"):
                out[c] = out[c].astype(object)
                out[c] = out[c].where(pd.notna(out[c]), np.nan)

            # If it's pandas nullable Int64/Float64 -> convert to float and replace <NA> with np.nan
            if str(out[c].dtype) in ("Int64", "Float64", "boolean"):
                out[c] = out[c].astype(float)
                out[c] = out[c].where(pd.notna(out[c]), np.nan)

        # Also: if any remaining pd.NA exist anywhere, force them to np.nan
        out = out.replace({pd.NA: np.nan})
        return out

    X_train_safe = sklearn_safe_df(X_train)
    X_test_safe  = sklearn_safe_df(X_test)

    clf_boost.fit(X_train_safe, y_train_i, model__sample_weight=w_train)
    pred_i = clf_boost.predict(X_test_safe)
    pred = pd.Series(pred_i).map(int_to_label).values

    print("=== Baseline Boosted Trees (multiclass) ===")
    print(classification_report(y_test, pred, digits=4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
    print("Confusion matrix labels:", label_order)
    print(confusion_matrix(y_test, pred, labels=label_order))
else:
    print("Skip boosted model (LightGBM not installed).")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001398 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1985
[LightGBM] [Info] Number of data points in the train set: 11559, number of used features: 52
[LightGBM] [Info] Start training from score -1.166489
[LightGBM] [Info] Start training from score -0.557454
[LightGBM] [Info] Start training from score -2.413256
[LightGBM] [Info] Start training from score -3.636166
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== Baseline Boosted Trees (multiclass) ===
               precision    recall  f1-score   support

           AD     0.7815    0.7841    0.7828       602
NoDementia_5y     0.9431    0.9639    0.9534      2217
       PD_LBD     0.6000    0.2069    0.3077        58
     Vascular     0.0000    0.0000    0.0000        13

     accuracy                         0.9069      2890
    macro avg     0.5811    0.4887    0.5110      2890
 weighted avg     0.8983    0.9069    0.9006      2890

Balanced accuracy: 0.48871622714800333
Confusion matrix labels: ['AD', 'NoDementia_5y', 'PD_LBD', 'Vascular']
[[ 472  122    8    0]
 [  80 2137    0    0]
 [  40    6   12    0]
 [  12    1    0    0]]


c:\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# -----------------------------
# Hierarchical model
# Stage 1: NoDementia vs Dementia-within-5y
# Stage 2: subtype among {AD, PD_LBD, Vascular} conditioned on dementia
# Combine: P(subtype) = P(dementia)*P(subtype|dementia); P(NoDem) = 1 - P(dementia)
# -----------------------------
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np
import pandas as pd
import os

if not HAS_LGBM:
    print("Hierarchical model requires LightGBM.")
else:
    # IMPORTANT: use sklearn-safe X (no pd.NA)
    X_train_safe = sklearn_safe_df(X_train)
    X_test_safe  = sklearn_safe_df(X_test)

    # -----------------------------
    # Stage 1: Binary Dementia within 5y?
    # -----------------------------
    y1_train = (y_train != "NoDementia_5y").astype(int)
    y1_test  = (y_test  != "NoDementia_5y").astype(int)

    # Give dementia positives higher weight
    w1_train = np.where(y1_train.values == 1, 3.0, 1.0).astype(np.float64)

    stage1 = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])

    stage1.fit(X_train_safe, y1_train, model__sample_weight=w1_train)
    p_dem = stage1.predict_proba(X_test_safe)[:, 1]  # P(dementia within 5y)

    pred1 = (p_dem >= 0.5).astype(int)

    print("\n=== Stage 1 (Binary): Dementia within 5y? ===")
    print(classification_report(y1_test, pred1, digits=4, target_names=["NoDementia", "Dementia"]))
    print("Balanced accuracy:", balanced_accuracy_score(y1_test, pred1))
    print("Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):")
    print(confusion_matrix(y1_test, pred1))

    # -----------------------------
    # -----------------------------
    # Stage 2a: Vascular vs Non-vascular among dementia cases only
    # -----------------------------
    dementia_mask_train = (y_train != "NoDementia_5y")
    X2_train = X_train.loc[dementia_mask_train].copy()
    y2_train = y_train.loc[dementia_mask_train].copy()
    X2_train_safe = sklearn_safe_df(X2_train)

    y_vas_train = (y2_train == "Vascular").astype(int)

    # Heavier weight on vascular positives
    w_vas = np.where(y_vas_train.values == 1, 25.0, 1.0).astype(np.float64)

    stage2a_vas = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])
    stage2a_vas.fit(X2_train_safe, y_vas_train, model__sample_weight=w_vas)

    # -----------------------------
    # Stage 2b: PD_LBD vs AD among NON-vascular dementia cases only
    # -----------------------------
    nonvas_mask_train = dementia_mask_train & (y_train != "Vascular")
    X3_train = X_train.loc[nonvas_mask_train].copy()
    y3_train = y_train.loc[nonvas_mask_train].copy()
    X3_train_safe = sklearn_safe_df(X3_train)

    # binary label: PD_LBD=1, AD=0
    bad = sorted(set(y3_train.unique()) - set(["AD", "PD_LBD"]))
    if bad:
        raise ValueError(f"Unexpected labels in non-vascular dementia subset: {bad}")

    y_pd_train = (y3_train == "PD_LBD").astype(int)

    # Weight PD positives a bit
    w_pd = np.where(y_pd_train.values == 1, 10.0, 1.0).astype(np.float64)

    stage2b_pd = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])
    stage2b_pd.fit(X3_train_safe, y_pd_train, model__sample_weight=w_pd)

    # -----------------------------
    # Stage 2 evaluation on TEST dementia cases (optional but useful)
    # -----------------------------
    dementia_mask_test = (y_test != "NoDementia_5y")
    X2_test = X_test.loc[dementia_mask_test].copy()
    y2_test = y_test.loc[dementia_mask_test].copy()
    X2_test_safe = sklearn_safe_df(X2_test)

    p_vas_cond = stage2a_vas.predict_proba(X2_test_safe)[:, 1]  # P(Vascular | Dementia)

    # For non-vascular true dementia cases, compute PD vs AD
    nonvas_test_mask = dementia_mask_test & (y_test != "Vascular")
    X3_test = X_test.loc[nonvas_test_mask].copy()
    y3_test = y_test.loc[nonvas_test_mask].copy()
    X3_test_safe = sklearn_safe_df(X3_test)

    p_pd_cond_nonvas = stage2b_pd.predict_proba(X3_test_safe)[:, 1]  # P(PD | Dementia, NonVas)

    # -----------------------------
    # Combine Stage1 + Stage2a + Stage2b into 4-class prediction
    # -----------------------------
    label_order_4 = ["NoDementia_5y", "AD", "PD_LBD", "Vascular"]
    probs = np.zeros((len(X_test_safe), 4), dtype=float)

    # Stage 2a and 2b probabilities for ALL test rows
    p_vas_all = stage2a_vas.predict_proba(X_test_safe)[:, 1]  # P(Vascular | Dementia) approx
    p_pd_all  = stage2b_pd.predict_proba(X_test_safe)[:, 1]   # P(PD | Dementia, NonVas) approx

    # Conditional subtype probabilities given dementia:
    # P(Vascular|Dem) = p_vas_all
    # P(PD|Dem) = (1 - p_vas_all) * p_pd_all
    # P(AD|Dem) = (1 - p_vas_all) * (1 - p_pd_all)
    p_ad_cond_all = (1.0 - p_vas_all) * (1.0 - p_pd_all)
    p_pd_cond_all = (1.0 - p_vas_all) * p_pd_all

    # Joint probabilities
    probs[:, 0] = 1.0 - p_dem
    probs[:, 3] = p_dem * p_vas_all
    probs[:, 2] = p_dem * p_pd_cond_all
    probs[:, 1] = p_dem * p_ad_cond_all

    # ---------- THRESHOLDS ----------
    T_DEM = 0.45
    T_VAS = 0.03
    T_PD  = 0.35
    #MARGIN_JOINT = 0.002


    pred_h = np.full(len(X_test_safe), "NoDementia_5y", dtype=object)

    is_dem = (p_dem >= T_DEM)
    idx = np.where(is_dem)[0]
    if len(idx) > 0:
        vas_ok = (p_vas_all[idx] >= T_VAS)
        MARGIN_JOINT = -0.20
        #vas_ok = (p_vas_all[idx] >= T_VAS) & ((probs[idx, 3] - probs[idx, 1]) >= MARGIN_JOINT)

        # default AD for dementia
        pred_sub = np.full(len(idx), "AD", dtype=object)

        # among non-vascular, decide PD vs AD
        pd_ok = (p_pd_all[idx] >= T_PD)
        pred_sub[pd_ok] = "PD_LBD"
        pred_sub[vas_ok] = "Vascular"  # vascular wins if it passes its gate

        pred_h[idx] = pred_sub

    print("\n=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===")
    print(classification_report(y_test, pred_h, digits=4, labels=label_order_4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_h))
    print("Confusion matrix labels:", label_order_4)
    print(confusion_matrix(y_test, pred_h, labels=label_order_4))

    # Save probabilities + predictions
    prob_df = pd.DataFrame(probs, columns=[f"p_{c}" for c in label_order_4], index=X_test.index)
    prob_df["p_dem"] = p_dem
    prob_df["p_ad_cond"] = p_ad_cond_all
    prob_df["p_pd_cond"] = p_pd_cond_all
    prob_df["p_vas_cond"] = p_vas_all
    prob_df["y_true"] = y_test.values
    prob_df["y_pred"] = pred_h
    prob_path = os.path.join(OUT_DIR, "hierarchical_pred_probs.csv")
    prob_df.to_csv(prob_path, index=True)
    print("Saved:", prob_path)


[LightGBM] [Info] Number of positive: 2693, number of negative: 8866
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002409 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1985
[LightGBM] [Info] Number of data points in the train set: 11559, number of used features: 52
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.476778 -> initscore=-0.092956
[LightGBM] [Info] Start training from score -0.092956


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Stage 1 (Binary): Dementia within 5y? ===
              precision    recall  f1-score   support

  NoDementia     0.9472    0.9540    0.9506      2217
    Dementia     0.8447    0.8247    0.8346       673

    accuracy                         0.9239      2890
   macro avg     0.8960    0.8893    0.8926      2890
weighted avg     0.9233    0.9239    0.9236      2890

Balanced accuracy: 0.8893287784987142
Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):
[[2115  102]
 [ 118  555]]
[LightGBM] [Info] Number of positive: 51, number of negative: 2642
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002368 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1168
[LightGBM] [Info] Number of data points in the train set: 2693, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.325504 -> initscore=-0.728590
[LightGBM] [Info] Start training from score -0.728590
[Ligh

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===
               precision    recall  f1-score   support

NoDementia_5y     0.9500    0.9504    0.9502      2217
           AD     0.7552    0.7791    0.7670       602
       PD_LBD     0.4474    0.2931    0.3542        58
     Vascular     0.1538    0.1538    0.1538        13

     accuracy                         0.8979      2890
    macro avg     0.5766    0.5441    0.5563      2890
 weighted avg     0.8957    0.8979    0.8965      2890

Balanced accuracy: 0.544100692639052
Confusion matrix labels: ['NoDementia_5y', 'AD', 'PD_LBD', 'Vascular']
[[2107  108    0    2]
 [ 105  469   21    7]
 [   5   34   17    2]
 [   1   10    0    2]]
Saved: ../output\hierarchical_pred_probs.csv


In [39]:
# # --- Diagnose subtype probabilities by true label ---
# vas_true = (y_test == "Vascular").values
# ad_true  = (y_test == "AD").values

# print("True Vascular count:", int(np.sum(vas_true)))
# print("p_vas stats for TRUE Vascular: min/median/max =",
#       float(np.min(p_vas[vas_true])),
#       float(np.median(p_vas[vas_true])),
#       float(np.max(p_vas[vas_true])))

# print("How many TRUE Vascular have p_vas >= 0.10?",
#       int(np.sum(p_vas[vas_true] >= 0.10)))
# print("How many TRUE Vascular have p_vas >= 0.05?",
#       int(np.sum(p_vas[vas_true] >= 0.05)))
# print("How many TRUE Vascular have p_vas >= 0.02?",
#       int(np.sum(p_vas[vas_true] >= 0.02)))


In [40]:
# print("Sanity counts in idx:",
#       ">=0.15:", int(np.sum(p_vas[idx] >= 0.15)),
#       ">=0.10:", int(np.sum(p_vas[idx] >= 0.10)))

In [41]:
# border = np.sum((p_vas[idx] >= 0.10) & (p_vas[idx] < 0.15))
# print("Borderline vascular inside idx (0.10–0.15):", int(border))


In [42]:
# # how many would newly pass if ONLY threshold mattered (no margin)
# new_by_thresh = np.sum((p_vas[idx] >= 0.10) & (p_vas[idx] < 0.15))
# print("New candidates unlocked by threshold (0.10–0.15):", int(new_by_thresh))

# # how many of those also pass the margin rule
# new_by_thresh_and_margin = np.sum(
#     (p_vas[idx] >= 0.10) & (p_vas[idx] < 0.15) & ((p_vas[idx] - p_ad[idx]) >= MARGIN_VAS)
# )
# print("…of those, pass margin too:", int(new_by_thresh_and_margin))


In [43]:
# print("T_VAS currently:", T_VAS)
# print("Among dementia-gated idx: n =", len(idx))
# print("vas_ok count:", int(np.sum(vas_ok)))
# print("pd_ok  count:", int(np.sum(pd_ok)))
# print("both   count:", int(np.sum(pd_ok & vas_ok)))

In [44]:
# print("Min/Median/Max p_vas:", np.min(p_vas), np.median(p_vas), np.max(p_vas))
# print("Count p_vas >= 0.25:", np.sum(p_vas >= 0.25))
# print("Count p_vas >= 0.15:", np.sum(p_vas >= 0.15))
# print("Count p_vas >= 0.10:", np.sum(p_vas >= 0.10))
# print("Count p_vas >= 0.05:", np.sum(p_vas >= 0.05))


In [51]:
print("T_VAS:", T_VAS, "MARGIN_JOINT:", MARGIN_JOINT)
print("Eligible by p_vas_all >= T_VAS:", int(np.sum(p_vas_all[idx] >= T_VAS)))

d = (probs[idx, 3] - probs[idx, 1])
eligible = (p_vas_all[idx] >= T_VAS)
print("Eligible by margin among eligible:", int(np.sum(eligible & (d >= MARGIN_JOINT))))
print("Joint diff stats among eligible: min/median/max =",
      float(np.min(d[eligible])) if np.any(eligible) else None,
      float(np.median(d[eligible])) if np.any(eligible) else None,
      float(np.max(d[eligible])) if np.any(eligible) else None)


T_VAS: 0.02 MARGIN_JOINT: -0.2
Eligible by p_vas_all >= T_VAS: 18
Eligible by margin among eligible: 4
Joint diff stats among eligible: min/median/max = -0.9561971687864785 -0.7139762078275586 0.35637051662795477


In [45]:

# -----------------------------
# Quick "which factors matter" (model-driven)
# For LightGBM: built-in gain importance on the multiclass baseline model
# -----------------------------
if HAS_LGBM:
    # Get feature names after preprocessing
    def get_feature_names(ct: ColumnTransformer):
        out = []
        # numeric
        out.extend(ct.named_transformers_["num"].get_feature_names_out(num_cols))
        # categorical
        ohe = ct.named_transformers_["cat"].named_steps["onehot"]
        cats = ohe.get_feature_names_out(cat_cols)
        out.extend(cats.tolist())
        return out

    feat_names = get_feature_names(clf_boost.named_steps["preprocess"])
    booster_model = clf_boost.named_steps["model"]

    # LightGBM importance is aggregated across trees/classes
    imp = pd.DataFrame({
        "feature": feat_names,
        "importance_gain": booster_model.feature_importances_
    }).sort_values("importance_gain", ascending=False)

    imp_path = os.path.join(OUT_DIR, "feature_importance_boosted.csv")
    imp.to_csv(imp_path, index=False)
    print("Saved:", imp_path)
    display(imp.head(25))


Saved: ../output\feature_importance_boosted.csv


,feature,importance_gain
32,max_follow_years,36670
26,TRAILB,31816
25,TRAILA,29973
0,NACCAGE,29044
27,ANIMALS,22186
1,EDUC,15020
28,BOSTON,14190
29,CDRSUM,13458
23,NACCMMSE,10869
24,NACCMOCA,8207


In [46]:

# -----------------------------
# Subgroup performance (sex/race/education bin)
# Note: subgroup counts for rare classes can be tiny -> interpret cautiously.
# -----------------------------
from sklearn.metrics import balanced_accuracy_score

def subgroup_report(predict_fn, X_test_df, y_test_ser, group_series, group_name):
    rows = []
    for g in sorted(group_series.dropna().unique()):
        mask = (group_series == g)
        if mask.sum() < 30:
            continue
        yt = y_test_ser[mask]
        yp = predict_fn(X_test_df.loc[mask])
        rows.append({
            "group_name": group_name,
            "group_value": str(g),
            "n": int(mask.sum()),
            "balanced_acc": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "recall_AD": ( (yp == "AD") & (yt == "AD") ).sum() / max(1, (yt == "AD").sum()),
            "recall_NoDementia": ( (yp == "NoDementia_5y") & (yt == "NoDementia_5y") ).sum() / max(1, (yt == "NoDementia_5y").sum()),
            "recall_PD_LBD": ( (yp == "PD_LBD") & (yt == "PD_LBD") ).sum() / max(1, (yt == "PD_LBD").sum()),
            "recall_Vascular": ( (yp == "Vascular") & (yt == "Vascular") ).sum() / max(1, (yt == "Vascular").sum()),
        })
    return pd.DataFrame(rows)

subgroups = {}
for col in ["SEX","RACE","EDUC_BIN"]:
    if col in X_test.columns:
        subgroups[col] = X_test[col]

tables = []
if HAS_LGBM:
    # predict wrapper for baseline boosted model
    def predict_boost(Xdf):
        Xdf_safe = sklearn_safe_df(Xdf)          # <-- key fix
        pi = clf_boost.predict(Xdf_safe)
        return pd.Series(pi).map(int_to_label).values


    for name, s in subgroups.items():
        tables.append(subgroup_report(predict_boost, X_test, y_test, s, name))
    sub_boost = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    p_boost = os.path.join(OUT_DIR, "subgroup_metrics_boosted.csv")
    sub_boost.to_csv(p_boost, index=False)
    print("Saved:", p_boost)
    display(sub_boost)


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have v

Saved: ../output\subgroup_metrics_boosted.csv


,group_name,group_value,n,balanced_acc,macro_f1,recall_AD,recall_NoDementia,recall_PD_LBD,recall_Vascular
0,SEX,1,1215,0.447989,0.474192,0.658621,1.0,0.133333,0.0
1,SEX,2,1675,0.418269,0.434852,0.673077,1.0,0.000000,0.0
2,RACE,1,2396,0.443022,0.474462,0.652087,1.0,0.120000,0.0
3,RACE,2,365,0.431818,0.439221,0.727273,1.0,0.000000,0.0
4,RACE,5,79,0.425000,0.429739,0.700000,1.0,0.000000,0.0
5,EDUC_BIN,<=12,461,0.445977,0.457144,0.717241,1.0,0.066667,0.0
6,EDUC_BIN,>12,2429,0.441542,0.475990,0.649891,1.0,0.116279,0.0
